In [ ]:
import random
import numpy as np
import pandas as pd

import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
dataset = load_dataset("emotion")

for split in dataset:
    print(f"{split}: {len(dataset[split])}")


label_names = dataset["train"].features["label"].names
print("Classes:", label_names)

for i in range(5):
    example = dataset["train"][i]
    text = example["text"]
    label = example["label"]

    print(f"\nExample {i+1}:")
    print("Text:", text)
    print("Label:", label, f"({label_names[label]})")

### Что классифицируется

В данной работе используется датасет `emotion`, содержащий короткие текстовые сообщения.

Задача — классификация текста по выражаемой в нём эмоции.  
Каждому тексту соответствует одна из шести категорий: sadness, joy, love, anger, fear, surprise.

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

texts = [dataset["train"][i]["text"] for i in range(3)]

for i, text in enumerate(texts):
    encoding = tokenizer(text)

    print(f"\n=== Example {i+1} ===")
    print("Text:", text)

    print("\nTokens:")
    print(tokenizer.convert_ids_to_tokens(encoding["input_ids"]))

    print("\nInput IDs:")
    print(encoding["input_ids"])

    print("\nAttention mask:")
    print(encoding["attention_mask"])

In [ ]:
encoding = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=10
)

print("Input IDs:")
print(encoding["input_ids"])

print("\nAttention mask:")
print(encoding["attention_mask"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)

    pred_class = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_class].item()

    return pred_class, confidence


for i in range(5):
    text = dataset["test"][i]["text"]
    true_label = dataset["test"][i]["label"]

    pred_label, confidence = predict(text)

    print(f"\nText: {text}")
    print(f"True: {true_label} ({label_names[true_label]})")
    print(f"Pred: {pred_label} ({label_names[pred_label]})")
    print(f"Confidence: {confidence:.4f}")

### Инференс готовой модели

Используется предобученная модель BERT без дополнительного fine-tuning на выбранном датасете.

Полученные предсказания в целом не соответствуют истинным меткам, так как модель не обучалась на задаче классификации эмоций. Классификационный слой инициализирован случайно, поэтому результаты близки к случайным.

Это демонстрирует необходимость fine-tuning модели на конкретной задаче.

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
model.to(device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
    seed=SEED
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print("Test results:", test_results)

In [ ]:
import os
os.makedirs("./artifacts", exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np

predictions = trainer.predict(tokenized_datasets["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

df_pred = pd.DataFrame({
    "text": dataset["test"]["text"],
    "true_label": true_labels,
    "pred_label": pred_labels,
    "confidence": np.max(predictions.predictions, axis=-1)
})

label_names = dataset["train"].features["label"].names
df_pred["true_emotion"] = df_pred["true_label"].apply(lambda x: label_names[x])
df_pred["pred_emotion"] = df_pred["pred_label"].apply(lambda x: label_names[x])

df_pred.to_csv("./artifacts/sample_predictions.csv", index=False)

print("Первые 10 предсказаний:")
display(df_pred.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix on Test Set")
plt.tight_layout()
plt.savefig("./artifacts/confusion_matrix.png", dpi=200)
plt.show()

### Оценка качества

На тестовой выборке модель показала следующие результаты:

- **Accuracy**: 0.93
- **F1-macro**: 0.8776

Качество можно считать хорошим для 6-классовой задачи классификации

### Матрица ошибок

Модель отлично распознаёт самые массовые классы — `sadness` (563 правильных из 581) и `joy` (663 правильных из 695).  
Хуже всего она справляется с классами `love` и `surprise`.

`love` часто путается с `joy` (30 случаев)

`surprise` часто предсказывается как `fear` (18 случаев)


### Вывод

Fine-tuning значительно улучшил качество по сравнению с нулевым инференсом предобученной BERT (где предсказания были почти случайными). Модель хорошо научилась различать эмоции, хотя всё ещё путает близкие классы